In [32]:
#Start

import pandas as pd, torch
import torch
import torchvision
from torchvision.transforms import v2
import numpy as np
from sklearn.model_selection import train_test_split 
import torch.nn as nn  
import matplotlib.pyplot as plt
import warnings

dataset = torchvision.datasets.CustomImageDataset(annotations_file="kartdatasett_1169/kartdatasett/etiketter.csv", img_dir="kartdatasett_1169")

train_dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

class ConvModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3,   
                        out_channels=1,   
                        kernel_size=3,   
                        stride=1)
        self.conv2 = nn.Conv2d(in_channels=1,
                      out_channels=1,
                      kernel_size=5,
                      stride=1)
        self.conv3 = nn.Conv2d(in_channels=1,
                      out_channels=1,
                      kernel_size=6,
                      stride=2)
        self.relu = nn.ReLU()
        self.linear = nn.Linear(27,9)
        self.softmax = nn.Softmax(dim=1)

    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.relu(x)
        x = self.linear(x)
        x = self.softmax(x)
        return x

model = ConvModel()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

def run_one_epoch():
    running_loss = 0
    last_loss = 0
    for i, data in enumerate(dataset):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        print(outputs)
        loss = loss_fn(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss = loss.item()
        if i % 300 == 299:
            last_loss = running_loss/300
            print(f"loss {last_loss}")
            running_loss = 0
    return last_loss

for epochs in range(12):
    model.train(True)
    avg_loss = run_one_epoch()
    model.eval()
        
        
        

conv_train1 = conv_filter(X_train)
print(conv_train1)


RuntimeError: The archive ILSVRC2012_devkit_t12.tar.gz is not present in the root directory or is corrupted. You need to download it externally and place it in kartdatasett_1169.